# 1주차. LLM Agent - Function Call, Tool Call 기본 구조

## 0. 목표

이번 주에는 모델이 직접 일정을 저장하는 것이 아니라, 일정 생성 도구를 어떤 인자로 호출할지 결정한다는 점을 관찰한다.

성취 기준:

- `tool_call`과 `tool_result`를 구분해 설명할 수 있다.
- `create_schedule`과 `list_schedules` trace를 찾아 사용자 요청과 맞는지 확인할 수 있다.
- 답변 문구보다 tool arguments와 저장 결과를 먼저 검증할 수 있다.


## 1. 준비

로컬 `Jupyter Notebook` 또는 `JupyterLab`에서 repo 루트를 기준으로 실행한다.

모든 실습은 실제 OpenAI API를 호출한다. API 키와 모델 설정은 repo 루트의 `.env`에서 읽는다.

처음 읽는 순서: `docs/orientation.md` → 이 노트북 → `week01.py`의 `# TODO 문제`를 먼저 풀고 바로 아래 `# 모범 답안`과 비교한 뒤 Gradio 데모를 실행한다.


In [9]:
# 흐름: repo 위치와 .env를 잡고, 모든 실습 셀이 함께 쓰는 모델/trace helper를 준비한다.
# 학생은 final_text보다 extract_tool_trace 결과를 먼저 확인한다.
import json
import os
import sys
from pathlib import Path
from typing import Any


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / ".env").exists() or (candidate / ".env.example").exists():
            return candidate
    raise RuntimeError("repo root를 찾지 못했습니다. repo 안에서 노트북을 실행하세요.")


REPO_ROOT = find_repo_root(Path.cwd())

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI

load_dotenv(REPO_ROOT / ".env", override=True)

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
OPENAI_EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("repo 루트 .env 파일에 OPENAI_API_KEY를 설정한 뒤 다시 실행하세요.")


def make_model(max_tokens: int = 500) -> ChatOpenAI:
    return ChatOpenAI(model=OPENAI_MODEL, temperature=0, max_completion_tokens=max_tokens)


def show_json(value: Any) -> None:
    print(json.dumps(value, ensure_ascii=False, indent=2, default=str))


def final_text(agent_result: dict[str, Any]) -> str:
    return agent_result["messages"][-1].content


def extract_tool_trace(agent_result: dict[str, Any]) -> list[dict[str, Any]]:
    trace = []
    for message in agent_result.get("messages", []):
        for call in getattr(message, "tool_calls", []) or []:
            trace.append({"event": "tool_call", "tool_name": call.get("name"), "arguments": call.get("args", {})})
        if getattr(message, "type", None) == "tool":
            trace.append({"event": "tool_result", "tool_name": getattr(message, "name", None), "content": message.content})
    return trace


## 2. 개념

오늘의 큰 그림: 자연어 요청이 들어오면 모델은 Python 함수를 직접 실행하지 않고, 호출할 도구 이름과 arguments를 구조화해서 만든다. 수업 코드는 그 tool call을 실행하고 tool result를 다시 모델에게 전달한다.

반드시 이해할 것:

- `tool_call`: 모델이 어떤 tool을 어떤 arguments로 호출하겠다고 정한 기록
- `tool_result`: 실제 Python tool 실행 결과
- 일정 생성 성공 여부는 최종 답변보다 trace와 저장 목록으로 확인한다.

지금은 몰라도 되는 것:

- LangChain agent 내부 메시지 루프
- tool schema가 모델 prompt로 변환되는 세부 방식

막혔을 때 볼 trace: `create_schedule`의 arguments, `list_schedules`의 result, `created_schedule`의 `id/date/start_time`.


## 3. 기본 개념 실습

가장 작은 흐름은 "자연어 요청 → 일정 생성 tool call → 저장된 일정 확인"이다. 여기서는 생성 도구 하나만 등록해 모델이 어떤 arguments를 만드는지 본다.


In [ ]:
# 흐름: 첫 번째 tool인 create_schedule을 만들고, agent가 이 tool을 호출하는지 확인한다.
# schedules 리스트는 실제 DB 대신 상태 변화를 눈으로 보기 위한 작은 저장소다.
schedules: list[dict[str, Any]] = []

@tool("create_schedule", description="개인 일정을 생성한다. date는 YYYY-MM-DD, start_time은 HH:MM 형식이다.")
def create_schedule(title: str, date: str, start_time: str, attendees: list[str] | None = None) -> str:
    """Create a personal schedule."""
    schedule = {"id": f"schedule-{len(schedules) + 1}", "title": title, "date": date, "start_time": start_time, "attendees": attendees or []}
    schedules.append(schedule)
    return json.dumps({"ok": True, "schedule": schedule}, ensure_ascii=False)

nana_basic_agent = create_agent(
    model=make_model(),
    tools=[create_schedule],
    system_prompt="너는 개인 일정 메이트 나나다. 오늘은 2026-04-23이다. 상대 날짜는 이 날짜 기준으로 YYYY-MM-DD로 바꾼다. 일정 생성이 필요하면 반드시 create_schedule 도구를 호출한 뒤 짧게 답한다.",
)

student_request = "내일 10시에 민수와 회의 일정 잡아줘"
result = nana_basic_agent.invoke({"messages": [{"role": "user", "content": student_request}]})

print(final_text(result))
show_json(extract_tool_trace(result))
show_json(schedules)


## 4. 카나메이트 확장 예제

기본 일정 생성에 `list_schedules` 조회 기능을 붙인다. 이제 나나는 일정을 만들고, 같은 저장소에서 현재 일정 목록도 읽을 수 있다.


In [ ]:
# 흐름: 두 번째 tool인 list_schedules를 추가해, 생성된 일정이 상태에 남았는지 확인한다.
# agent가 가진 tool 목록이 늘어나면 모델이 상황에 맞게 tool을 선택한다.
@tool("list_schedules", description="현재 생성된 개인 일정 목록을 조회한다.")
def list_schedules() -> str:
    """List personal schedules."""
    return json.dumps({"ok": True, "schedules": schedules}, ensure_ascii=False)

nana_agent = create_agent(
    model=make_model(),
    tools=[create_schedule, list_schedules],
    system_prompt="너는 개인 일정 메이트 나나다. 오늘은 2026-04-23이다. 상대 날짜는 이 날짜 기준으로 YYYY-MM-DD로 바꾼다. 일정 생성이나 조회가 필요하면 반드시 알맞은 도구를 호출한 뒤 짧게 답한다.",
)

extension_request = "지금까지 만든 일정 목록 보여줘"
extension_result = nana_agent.invoke({"messages": [{"role": "user", "content": extension_request}]})

print(final_text(extension_result))
show_json(extract_tool_trace(extension_result))


현재까지 만든 일정 목록은 다음과 같습니다:

- **제목**: 회의
  - **날짜**: 2026-04-24
  - **시간**: 10:00
  - **참석자**: 민수
[
  {
    "event": "tool_call",
    "tool_name": "list_schedules",
    "arguments": {}
  },
  {
    "event": "tool_result",
    "tool_name": "list_schedules",
    "content": "{\"ok\": true, \"schedules\": [{\"id\": \"schedule-1\", \"title\": \"회의\", \"date\": \"2026-04-24\", \"start_time\": \"10:00\", \"attendees\": [\"민수\"]}]}"
  }
]


## 5. 확장 예제 실행

같은 agent로 새 일정을 하나 더 만들고, 곧바로 목록 조회를 실행한다. 수강생은 요청 문장을 바꿔보며 생성 trace와 조회 trace가 어떻게 달라지는지 확인한다.


In [ ]:
# 흐름: 생성 요청과 목록 조회 요청을 연달아 실행해 tool call trace와 저장 상태를 비교한다.
# assert는 모델 문구가 아니라 tool_result payload가 맞는지 점검한다.
extension_create_request = "내일 14시에 지아와 체크인 일정 잡아줘"
extension_create_result = nana_agent.invoke({"messages": [{"role": "user", "content": extension_create_request}]})
extension_create_trace = extract_tool_trace(extension_create_result)

extension_list_request = "현재 일정 목록 보여줘"
extension_list_result = nana_agent.invoke({"messages": [{"role": "user", "content": extension_list_request}]})
extension_list_trace = extract_tool_trace(extension_list_result)

print(final_text(extension_create_result))
show_json(extension_create_trace)
print(final_text(extension_list_result))
show_json(extension_list_trace)

assert any(event["event"] == "tool_call" and event["tool_name"] == "create_schedule" for event in extension_create_trace)
assert any(event["event"] == "tool_call" and event["tool_name"] == "list_schedules" for event in extension_list_trace)
assert len(schedules) >= 1
print("1주차 확장 예제 실행 통과")


내일 2026-04-24 14시에 지아와 체크인 일정을 잡았습니다.
[
  {
    "event": "tool_call",
    "tool_name": "create_schedule",
    "arguments": {
      "title": "체크인",
      "date": "2026-04-24",
      "start_time": "14:00",
      "attendees": [
        "지아"
      ]
    }
  },
  {
    "event": "tool_result",
    "tool_name": "create_schedule",
    "content": "{\"ok\": true, \"schedule\": {\"id\": \"schedule-2\", \"title\": \"체크인\", \"date\": \"2026-04-24\", \"start_time\": \"14:00\", \"attendees\": [\"지아\"]}}"
  }
]
현재 일정 목록은 다음과 같습니다:

1. **회의**
   - 날짜: 2026-04-24
   - 시작 시간: 10:00
   - 참석자: 민수

2. **체크인**
   - 날짜: 2026-04-24
   - 시작 시간: 14:00
   - 참석자: 지아
[
  {
    "event": "tool_call",
    "tool_name": "list_schedules",
    "arguments": {}
  },
  {
    "event": "tool_result",
    "tool_name": "list_schedules",
    "content": "{\"ok\": true, \"schedules\": [{\"id\": \"schedule-1\", \"title\": \"회의\", \"date\": \"2026-04-24\", \"start_time\": \"10:00\", \"attendees\": [\"민수\"]}, {\"id\": \"schedule-2\", 

## 6. Week 1 독립 실행

노트북에서는 `.py` 파일을 import해서 실행하지 않는다. 1주차 독립 실행 데모는 터미널에서 repo 루트 기준으로 실행한다.

```bash
conda activate langchain
python week01.py
```

OpenAI quota가 부족할 때 1주차 흐름만 로컬 fallback으로 확인하려면 다음처럼 실행한다.

```bash
KANAMATE_OFFLINE=1 python week01.py
```


## 7. 회고

개념 확인 질문:

1. `tool_call`과 `tool_result`는 각각 누가 만들고 무엇을 의미하는가?
2. 사용자가 "내일 10시"라고 말했을 때 사람이 검증해야 할 arguments는 무엇인가?
3. 최종 답변이 자연스러워도 `list_schedules` trace를 확인해야 하는 이유는 무엇인가?

작은 응용 과제: 참석자가 없는 개인 일정 요청과 참석자가 있는 일정 요청을 각각 실행하고, `attendees` arguments가 어떻게 달라지는지 비교한다.
